# Play from a screenshot

Read a real Wordfeud board + rack from a screenshot, then pick the best move
with a policy. Uses the **learned** agent if a saved model exists, otherwise
falls back to the greedy (best-scoring) policy — currently our only ready model.

> Caveats: letter recognition has no confidence threshold yet, so empty tiles
> may show phantom letters; and confirm the bonus-code layout matches your board.

In [1]:
import os
import matplotlib.pyplot as plt

from src.screenshot_to_map import WordfeudMap
from src.dictionary import load_dictionary
from src.move_generator import WordfeudEngine, split_board, plot_board
from src.env import choose_action, CAND_FEATURES
from src.eval import best_candidate_policy

## 1. Read the screenshot → board + rack

In [ ]:
path = "imgs/frame.jpeg"

WF = WordfeudMap(path)
# Blank-aware read: letters (lowercase), bonus codes, blank positions, rack ('*').
letters, bonus, board_blanks, rack = WF.read_state()
print("rack:", rack)
print("blank tiles on board:", sorted(board_blanks))

## 2. Build the engine and pick the policy
**3-ply lookahead**: for each candidate it simulates the opponent's reply *and*
your own next turn, over racks sampled from the unseen-tile pool —
`value = your_score − E[opponent reply] + E[your next turn]`. This plays defense
(avoids opening premium squares) and values your rack leave, with no training.
Once the bag empties the opponent's rack is known, so it switches to **exact**
endgame evaluation. `n_samples` / `max_candidates` / `plies` trade accuracy for
speed (3-ply over a 30-move pool takes ~20 s).

In [ ]:
import random
from src.lookahead import LookaheadAgent

words = [w for lst in load_dictionary().values() for w in lst]
engine = WordfeudEngine(words)

# plies=3 for strength; drop to 2 (and/or lower n_samples) if you want it faster.
agent = LookaheadAgent(n_samples=8, max_candidates=12, plies=3, rng=random.Random(0))
print("using 3-ply Monte-Carlo lookahead (exact in the endgame)")

## 3. Top 10 moves (ranked)
A ranked fallback list of **distinct words** — if the official Wordfeud app
rejects the #1 word (its dictionary may differ from ours), try the next.
`value` = lookahead score net of the opponent's expected reply; `score` = raw
points; `E[opp]` = opponent's expected best reply.

In [ ]:
top = agent.top_moves(engine, letters, bonus, rack, n=10, board_blanks=board_blanks)

print(f"{'#':>2}  {'word':<12} {'dir':<3} {'start':<8} {'score':>5} {'value':>6} {'E[opp]':>6}")
print("-" * 50)
for t in top:
    print(f"{t['rank']:>2}  {t['word']:<12} {t['direction']:<3} "
          f"{str(t['start']):<8} {t['score']:>5} {t['value']:>6} {t['exp_opp']:>6}")

print("\nTo play #1 in the app, place these tiles (row, col, letter):")
for r, c, ch in top[0]["tiles"]:
    print(f"  ({r:2d}, {c:2d})  {ch.upper()}")

## 4. Visualise the ranked moves
Each board highlights one candidate's tiles, in rank order.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(24, 10))
for t, ax in zip(top, axes.ravel()):
    plot_board(letters, bonus, move=t["move"], ax=ax,
               title=f"#{t['rank']}  {t['word']} ({t['direction']})  +{t['score']}")
for ax in axes.ravel()[len(top):]:
    ax.axis("off")
plt.tight_layout()
plt.show()